In [53]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR 
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor,GradientBoostingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score,root_mean_squared_error
from sklearn.model_selection import RandomizedSearchCV

In [8]:
df=pd.read_csv('data/stud.csv')

In [9]:
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [12]:
X=df.drop(columns='math_score')
y=df['math_score']

In [47]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   gender                       1000 non-null   str  
 1   race_ethnicity               1000 non-null   str  
 2   parental_level_of_education  1000 non-null   str  
 3   lunch                        1000 non-null   str  
 4   test_preparation_course      1000 non-null   str  
 5   reading_score                1000 non-null   int64
 6   writing_score                1000 non-null   int64
dtypes: int64(2), str(5)
memory usage: 54.8 KB


In [14]:
y

0      72
1      69
2      90
3      47
4      76
       ..
995    88
996    62
997    59
998    68
999    77
Name: math_score, Length: 1000, dtype: int64

In [15]:
df.nunique()

gender                          2
race_ethnicity                  5
parental_level_of_education     6
lunch                           2
test_preparation_course         2
math_score                     81
reading_score                  72
writing_score                  77
dtype: int64

In [27]:
df[(df['parental_level_of_education'] == "master's degree") & (df['math_score'] > 80)]

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
2,female,group B,master's degree,standard,none,90,95,93
106,female,group D,master's degree,standard,none,87,100,100
128,male,group D,master's degree,standard,none,82,82,74
130,male,group D,master's degree,standard,none,89,84,82
164,female,group E,master's degree,standard,none,81,92,91
175,female,group C,master's degree,standard,completed,81,91,87
377,female,group D,master's degree,free/reduced,completed,85,95,100
604,male,group D,master's degree,free/reduced,completed,84,89,90
618,male,group D,master's degree,standard,none,95,81,84
685,female,group E,master's degree,standard,completed,94,99,100


In [18]:
df['parental_level_of_education'].value_counts()

parental_level_of_education
some college          226
associate's degree    222
high school           196
some high school      179
bachelor's degree     118
master's degree        59
Name: count, dtype: int64

In [ ]:


# Combined view
df.groupby(['parental_level_of_education', 'test_preparation_course'])['math_score'].mean()

parental_level_of_education  test_preparation_course
associate's degree           completed                  71.829268
                             none                       65.571429
bachelor's degree            completed                  73.282609
                             none                       66.902778
high school                  completed                  65.000000
                             none                       60.992857
master's degree              completed                  70.600000
                             none                       69.307692
some college                 completed                  71.454545
                             none                       64.892617
some high school             completed                  66.701299
                             none                       61.078431
Name: math_score, dtype: float64

Parental level of education is an ordinal ranking its values are differing 

In [29]:
df.groupby('parental_level_of_education')['math_score'].mean().sort_values()

parental_level_of_education
high school           62.137755
some high school      63.497207
some college          67.128319
associate's degree    67.882883
bachelor's degree     69.389831
master's degree       69.745763
Name: math_score, dtype: float64

In [30]:
### ORDINAL ENCODING needed 

In [44]:
num_features=X.select_dtypes(include='int64').columns
cat_features=X.select_dtypes(exclude='int64').columns
cat_features

Index(['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch',
       'test_preparation_course'],
      dtype='str')

In [45]:
cat_features=cat_features.drop('parental_level_of_education')

In [46]:
od_feature=['parental_level_of_education']

In [48]:
from sklearn.preprocessing import OneHotEncoder,StandardScaler,OrdinalEncoder
from sklearn.compose import ColumnTransformer

cat_orders=[['some high school', 'high school', 'some college', 
                      "associate's degree", "bachelor's degree", "master's degree"]]

preprocessor=ColumnTransformer(
    transformers=[('ohe',OneHotEncoder(drop='first'),cat_features),
                  ('od',OrdinalEncoder(categories=cat_orders),od_feature),
                  ('scaler',StandardScaler(),num_features)],
                  remainder='passthrough'
)

In [49]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [50]:
X_train=preprocessor.fit_transform(X_train)
X_test=preprocessor.transform(X_test)

In [75]:
def evalutaion(y_test,y_pred):
    mae = mean_absolute_error(y_test,y_pred)
    mse = mean_squared_error(y_test,y_pred)
    rmse = root_mean_squared_error(y_test,y_pred)
    r2 = r2_score(y_test,y_pred)
    return mae,mse,rmse,r2

In [76]:
models = {
    'Linear R': LinearRegression(),
    'Lasso R' : Lasso(),
    'Ridge R' : Ridge(),
    'KN Reg' : KNeighborsRegressor(),
    'DT R' : DecisionTreeRegressor(),
    'RF R' : RandomForestRegressor(),
    'Ada R': AdaBoostRegressor(),
    'Grad R': GradientBoostingRegressor(),
    'XG R' : XGBRegressor(),
    'SVR' : SVR()
}

In [77]:
best={}
for name,model in models.items():
    model.fit(X_train,y_train)
    y_train_pred=model.predict(X_train)
    y_test_pred=model.predict(X_test)

    print(f"TRAINING EVALUATION : {name}")
    mae,mse,rmse,r2=evalutaion(y_train,y_train_pred)
    print(f"mae : {mae}, mse : {mse}, rmse : {rmse}, r2 : {r2}")

    print(f"TESTING EVALUATION : {name}")
    mae,mse,rmse,r2=evalutaion(y_test,y_test_pred)
    print(f"mae : {mae}, mse : {mse}, rmse : {rmse}, r2 : {r2}")
    best[name]=r2


TRAINING EVALUATION : Linear R
mae : 4.280355358943515, mse : 28.483813890131685, rmse : 5.337022942627443, r2 : 0.8736565467932157
TESTING EVALUATION : Linear R
mae : 4.181966418321511, mse : 28.82105656383288, rmse : 5.36852461704637, r2 : 0.8815597679452446
TRAINING EVALUATION : Lasso R
mae : 5.205262535803302, mse : 43.46112340691518, rmse : 6.592505093431114, r2 : 0.8072228518043277
TESTING EVALUATION : Lasso R
mae : 5.155717042878383, mse : 42.475783470209954, rmse : 6.517344817501216, r2 : 0.8254456202958105
TRAINING EVALUATION : Ridge R
mae : 4.279096664391439, mse : 28.488390462491203, rmse : 5.3374516824502685, r2 : 0.8736362468446912
TESTING EVALUATION : Ridge R
mae : 4.183065515778031, mse : 28.81979857205805, rmse : 5.368407452127498, r2 : 0.8815649376668128
TRAINING EVALUATION : KN Reg
mae : 4.5222500000000005, mse : 31.901550000000007, rmse : 5.648145713417812, r2 : 0.8584967587137168
TESTING EVALUATION : KN Reg
mae : 5.811, mse : 55.91219999999999, rmse : 7.477446088070

In [78]:
best_score=pd.DataFrame(data=list(best.items()),columns=['name','score'])

In [79]:
best_score.groupby('name')['score'].max().sort_values(ascending=False)

name
Ridge R     0.881565
Linear R    0.881560
Grad R      0.873851
RF R        0.849866
Ada R       0.845442
XG R        0.828281
Lasso R     0.825446
KN Reg      0.770229
SVR         0.741455
DT R        0.740670
Name: score, dtype: float64

In [80]:
# best model is regression 